In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("/content/clean_ball_by_ball.csv")

print(" Dataset loaded successfully")
print("➡ Shape:", df.shape)
print("➡ Columns:",(df.columns))

/tmp/ipython-input-948736508.py:5: DtypeWarning: Columns (20,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/clean_ball_by_ball.csv")


 Dataset loaded successfully
➡ Shape: (197559, 33)
➡ Columns: Index(['matchid', 'inning', 'over_ball', 'over', 'ball', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'batsman_runs',
       'extras', 'iswide', 'isnoball', 'byes', 'legbyes', 'penalty',
       'dismissal_kind', 'player_dismissed', 'date', 'season', 'venue', 'city',
       'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'winner_runs', 'winner_wickets', 'neutralvenue', 'eliminator',
       'player_of_match'],
      dtype='object')


In [24]:
print("\nConverting date column to datetime...")

df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(" Date conversion completed")
print("➡ Date range:", df["date"].min(), "to", df["date"].max())



Converting date column to datetime...
 Date conversion completed
➡ Date range: 2008-04-18 00:00:00 to 2021-04-23 00:00:00


In [25]:
print("\nCreating helper flags (legal ball & wicket)...")

# 'is_legal_ball' indicates if a ball is a legitimate delivery (not a wide or no-ball)
df["is_legal_ball"] = (
    (df["iswide"] == 0) &
    (df["isnoball"] == 0)
).astype(int)

print("➡ 'is_legal_ball' column created")

# 'is_wicket' indicates if a dismissal occurred on that ball (excluding 'not_out')
df["is_wicket"] = (
    df["dismissal_kind"].notna() &
    (df["dismissal_kind"] != "not_out")
).astype(int)

print("➡ 'is_wicket' column created")
print(" Helper flags created")
print(f"➡ Legal balls count: {df["is_legal_ball"].sum()}")
print(f"➡ Total wickets: {df["is_wicket"].sum()}")

print("\nDisplaying DataFrame head with newly created helper flags:")
display(df[['inning', 'over_ball', 'iswide', 'isnoball', 'dismissal_kind', 'is_legal_ball', 'is_wicket']].head())


Creating helper flags (legal ball & wicket)...
➡ 'is_legal_ball' column created
➡ 'is_wicket' column created
 Helper flags created
➡ Legal balls count: 190777
➡ Total wickets: 9735

Displaying DataFrame head with newly created helper flags:


,inning,over_ball,iswide,isnoball,dismissal_kind,is_legal_ball,is_wicket
0,1,0.1,0,0,not_out,1,0
1,1,0.2,0,0,not_out,1,0
2,1,0.3,1,0,not_out,0,0
3,1,0.4,0,0,not_out,1,0
4,1,0.5,0,0,not_out,1,0


#  PLAYER–MATCH LEVEL DATASET CREATION

In [26]:
print("Creating batsman–match dataset")

batsman_match = (
    df
    .groupby(
        ["matchid", "date", "season", "venue", "city",
         "batting_team", "bowling_team", "batsman"],
        as_index=False
    )
    .agg(
        runs=("batsman_runs", "sum"),
        balls_faced=("is_legal_ball", "sum"),
        fours=("batsman_runs", lambda x: (x == 4).sum()),
        sixes=("batsman_runs", lambda x: (x == 6).sum())
    )
)

batsman_match["strike_rate"] = np.where(
    batsman_match["balls_faced"] > 0,
    batsman_match["runs"] / batsman_match["balls_faced"] * 100,
    0
)

print(" Batsman–match dataset created")
print("➡ Shape:", batsman_match.shape)
print("➡ Number of unique batsmen:", batsman_match['batsman'].nunique())
print("➡ Sample rows:")
print(batsman_match.head(5))
print("\n➡ Summary statistics:")
print(batsman_match.describe())


Creating batsman–match dataset
 Batsman–match dataset created
➡ Shape: (11680, 13)
➡ Number of unique batsmen: 540
➡ Sample rows:
   matchid       date   season                  venue       city  \
0   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
1   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
2   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
3   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
4   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   

            batting_team                 bowling_team          batsman  runs  \
0  Kolkata Knight Riders  Royal Challengers Bangalore      BB McCullum   158   
1  Kolkata Knight Riders  Royal Challengers Bangalore        DJ Hussey    12   
2  Kolkata Knight Riders  Royal Challengers Bangalore  Mohammad Hafeez     5   
3  Kolkata Knight Riders  Royal Challengers Bangalore       RT Ponting    20   
4  Kolkata Knight Riders  Royal Challengers Bangalore       S

In [27]:
print("Creating bowler–match level dataset...")

bowler_match = (
    df
    .groupby(
        ["matchid", "date", "season", "venue", "city",
         "bowling_team", "batting_team", "bowler"],
        as_index=False
    )
    .agg(
        runs_conceded=("batsman_runs", "sum"),
        balls_bowled=("is_legal_ball", "sum"),
        wicket=("is_wicket", "sum"),
        wides=("iswide", "sum"),
        no_balls=("isnoball", "sum")
    )
)

bowler_match["overs"] = bowler_match["balls_bowled"] / 6
bowler_match["economy"] = np.where(
    bowler_match["overs"] > 0,
    bowler_match["runs_conceded"] / bowler_match["overs"],
    0
)

print("Bowler–match dataset created")
print("➡ Shape:", bowler_match.shape)
print("➡ Number of unique bowlers:", bowler_match['bowler'].nunique())
print("➡ Sample rows:")
print(bowler_match.head(5))
print("\n➡ Summary statistics:")
print(bowler_match.describe())

Creating bowler–match level dataset...
Bowler–match dataset created
➡ Shape: (9283, 15)
➡ Number of unique bowlers: 426
➡ Sample rows:
   matchid       date   season                  venue       city  \
0   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
1   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
2   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
3   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   
4   335982 2008-04-18  2007/08  M Chinnaswamy Stadium  Bangalore   

            bowling_team                 batting_team      bowler  \
0  Kolkata Knight Riders  Royal Challengers Bangalore  AB Agarkar   
1  Kolkata Knight Riders  Royal Challengers Bangalore    AB Dinda   
2  Kolkata Knight Riders  Royal Challengers Bangalore    I Sharma   
3  Kolkata Knight Riders  Royal Challengers Bangalore   LR Shukla   
4  Kolkata Knight Riders  Royal Challengers Bangalore  SC Ganguly   

   runs_conceded  balls_bowled  wicket  w

In [28]:
print("Sorting datasets by player and date...")

batsman_match.sort_values(["batsman","date"], inplace=True)
bowler_match.sort_values(["bowler", "date"], inplace=True)

print("Sorting completed.")

Sorting datasets by player and date...
Sorting completed.


In [29]:
print(batsman_match.head(5))

      matchid       date season                                      venue  \
4301   548346 2012-04-29   2012                           Wankhede Stadium   
4400   548352 2012-05-04   2012            MA Chidambaram Stadium, Chepauk   
4498   548359 2012-05-08   2012  Rajiv Gandhi International Stadium, Uppal   
4701   548373 2012-05-18   2012  Rajiv Gandhi International Stadium, Uppal   
4749   548376 2012-05-20   2012  Rajiv Gandhi International Stadium, Uppal   

           city     batting_team                 bowling_team         batsman  \
4301     Mumbai  Deccan Chargers               Mumbai Indians  A Ashish Reddy   
4400    Chennai  Deccan Chargers          Chennai Super Kings  A Ashish Reddy   
4498  Hyderabad  Deccan Chargers              Kings XI Punjab  A Ashish Reddy   
4701  Hyderabad  Deccan Chargers             Rajasthan Royals  A Ashish Reddy   
4749  Hyderabad  Deccan Chargers  Royal Challengers Bangalore  A Ashish Reddy   

      runs  balls_faced  fours  sixes  strik

# RECENT FORM FEATURES

In [30]:
print("Creating batsman recent form features")

batsman_match["avg_runs_last_5"] = (
    batsman_match.groupby("batsman")["runs"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

batsman_match["avg_runs_last_10"] = (
    batsman_match.groupby("batsman")["runs"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

print("➡ Batsman recent form features created")
print("\nDisplaying head of batsman_match with new features:")
print(batsman_match.head(5))
print("\n")

print("Creating bowler recent form features...")

bowler_match["avg_wkts_last_5"] = (
    bowler_match.groupby("bowler")["wicket"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

bowler_match["avg_wkts_last_10"] = (
    bowler_match.groupby("bowler")["wicket"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

print("➡ Bowler recent form features created")
print("\nDisplaying head of bowler_match with new features:")
print(bowler_match.head(5))

Creating batsman recent form features
➡ Batsman recent form features created

Displaying head of batsman_match with new features:
      matchid       date season                                      venue  \
4301   548346 2012-04-29   2012                           Wankhede Stadium   
4400   548352 2012-05-04   2012            MA Chidambaram Stadium, Chepauk   
4498   548359 2012-05-08   2012  Rajiv Gandhi International Stadium, Uppal   
4701   548373 2012-05-18   2012  Rajiv Gandhi International Stadium, Uppal   
4749   548376 2012-05-20   2012  Rajiv Gandhi International Stadium, Uppal   

           city     batting_team                 bowling_team         batsman  \
4301     Mumbai  Deccan Chargers               Mumbai Indians  A Ashish Reddy   
4400    Chennai  Deccan Chargers          Chennai Super Kings  A Ashish Reddy   
4498  Hyderabad  Deccan Chargers              Kings XI Punjab  A Ashish Reddy   
4701  Hyderabad  Deccan Chargers             Rajasthan Royals  A Ashish Reddy

# VENUE-SPECIFIC FEATURES

In [31]:
print("Creating batsman venue-specific features")

batsman_match["avg_runs_at_venue"] = (
    batsman_match.groupby(["batsman", "venue"])["runs"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

batsman_match["matches_at_venue"] = (
    batsman_match.groupby(["batsman", "venue"]).cumcount()
)

print("\nCreating bowler venue-specific features")

bowler_match["avg_wkts_at_venue"] = (
    bowler_match.groupby(["bowler", "venue"])["wicket"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

bowler_match["matches_at_venue"] = (
    bowler_match.groupby(["bowler", "venue"]).cumcount()
)
print("\n",bowler_match.sample(5))

Creating batsman venue-specific features

Creating bowler venue-specific features

       matchid       date season                                       venue  \
5378   829717 2015-04-12   2015                            Wankhede Stadium   
5698   829773 2015-05-02   2015   Rajiv Gandhi International Stadium, Uppal   
4651   598064 2013-05-06   2013  Punjab Cricket Association Stadium, Mohali   
3891   597999 2013-04-04   2013                       M Chinnaswamy Stadium   
4088   598016 2013-04-16   2013  Punjab Cricket Association Stadium, Mohali   

            city                 bowling_team                 batting_team  \
5378      Mumbai              Kings XI Punjab               Mumbai Indians   
5698   Hyderabad          Sunrisers Hyderabad          Chennai Super Kings   
4651  Chandigarh  Royal Challengers Bangalore              Kings XI Punjab   
3891   Bangalore               Mumbai Indians  Royal Challengers Bangalore   
4088  Chandigarh        Kolkata Knight Riders      

In [32]:
print("\n[FEATURE ENGINEERING] Creating simple, interpretable batsman features...")


# 1. Matches Played (Experience)


batsman_match["matches_played"] = (
    batsman_match.groupby("batsman").cumcount()
)

print("✔ matches_played created")


# 2. Career Average Runs per Match


batsman_match["career_avg_runs"] = (
    batsman_match.groupby("batsman")["runs"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print("✔ career_avg_runs created")


# 3. Career Average Balls Faced per Match


batsman_match["career_avg_balls"] = (
    batsman_match.groupby("batsman")["balls_faced"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print("✔ career_avg_balls created")

# 4. Boundary Rate


batsman_match["boundary_rate"] = (
    (batsman_match["fours"] + batsman_match["sixes"]) /
    batsman_match["balls_faced"].replace(0, np.nan)
)

print("✔ boundary_rate created")


# 5. Recent vs Career Form Indicator


batsman_match["recent_form_indicator"] = (
    batsman_match["avg_runs_last_5"] >
    batsman_match["career_avg_runs"]
).astype(int)

print("✔ recent_form_indicator created")


# 6. Experience Bucket

batsman_match["experience_level"] = pd.cut(
    batsman_match["matches_played"],
    bins=[-1, 20, 50, 1000],
    labels=["new", "experienced", "veteran"]
)

print("✔ experience_level created")

# 7. Handle Missing Values

batsman_match.fillna({
    "career_avg_runs": 0,
    "career_avg_balls": 0,
    "boundary_rate": 0
}, inplace=True)

print("✔ Missing values handled")

print(" Simple batsman feature engineering completed")


[FEATURE ENGINEERING] Creating simple, interpretable batsman features...
✔ matches_played created
✔ career_avg_runs created
✔ career_avg_balls created
✔ boundary_rate created
✔ recent_form_indicator created
✔ experience_level created
✔ Missing values handled
 Simple batsman feature engineering completed


In [33]:
# ============================================================
# SIMPLE & EXPLAINABLE BOWLER FEATURE ENGINEERING
# ============================================================

import numpy as np

print("\n[FEATURE ENGINEERING] Creating simple, interpretable bowler features...")

# 1. Matches Played (Experience)

bowler_match["matches_played"] = (
    bowler_match.groupby("bowler").cumcount()
)

print("✔ matches_played created")

# 2. Career Average Wickets per Match

bowler_match["career_avg_wickets"] = (
    bowler_match.groupby("bowler")["wicket"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print("✔ career_avg_wickets created")

# 3. Career Average Runs Conceded per Match

bowler_match["career_avg_runs_conceded"] = (
    bowler_match.groupby("bowler")["runs_conceded"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print("✔ career_avg_runs_conceded created")

# 4. Career Average Overs per Match

bowler_match["career_avg_overs"] = (
    bowler_match.groupby("bowler")["overs"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print("✔ career_avg_overs created")

# 5. Discipline Score (Simple & Intuitive)

bowler_match["discipline_score"] = (
    (bowler_match["wides"] + bowler_match["no_balls"]) /
    bowler_match["balls_bowled"].replace(0, np.nan)
)

print("✔ discipline_score created")

# 6. Recent vs Career Form Indicator

bowler_match["recent_vs_career_form"] = (
    bowler_match["avg_wkts_last_5"] >
    bowler_match["career_avg_wickets"]
).astype(int)

print("✔ recent_vs_career_form created")

# 7. Handle Missing Values (Early Career)

bowler_match.fillna({
    "career_avg_wickets": 0,
    "career_avg_runs_conceded": 0,
    "career_avg_overs": 0,
    "discipline_score": 0
}, inplace=True)

print("✔ Missing values handled")

print("✅ Simple bowler feature engineering completed")



[FEATURE ENGINEERING] Creating simple, interpretable bowler features...
✔ matches_played created
✔ career_avg_wickets created
✔ career_avg_runs_conceded created
✔ career_avg_overs created
✔ discipline_score created
✔ recent_vs_career_form created
✔ Missing values handled
✅ Simple bowler feature engineering completed


In [34]:
print("Name of columns in batsman dataset : ",batsman_match.columns)
print("Name of columns in bowler dataset : ",bowler_match.columns)
print("Shape of batsman dataset : ",batsman_match.shape)
print("Shape of bowler dataset : ",bowler_match.shape)

Name of columns in batsman dataset :  Index(['matchid', 'date', 'season', 'venue', 'city', 'batting_team',
       'bowling_team', 'batsman', 'runs', 'balls_faced', 'fours', 'sixes',
       'strike_rate', 'avg_runs_last_5', 'avg_runs_last_10',
       'avg_runs_at_venue', 'matches_at_venue', 'matches_played',
       'career_avg_runs', 'career_avg_balls', 'boundary_rate',
       'recent_form_indicator', 'experience_level'],
      dtype='object')
Name of columns in bowler dataset :  Index(['matchid', 'date', 'season', 'venue', 'city', 'bowling_team',
       'batting_team', 'bowler', 'runs_conceded', 'balls_bowled', 'wicket',
       'wides', 'no_balls', 'overs', 'economy', 'avg_wkts_last_5',
       'avg_wkts_last_10', 'avg_wkts_at_venue', 'matches_at_venue',
       'matches_played', 'career_avg_wickets', 'career_avg_runs_conceded',
       'career_avg_overs', 'discipline_score', 'recent_vs_career_form'],
      dtype='object')
Shape of batsman dataset :  (11680, 23)
Shape of bowler dataset : 

In [35]:
# HEALTH REPORT FUNCTION
def dataset_audit(df, name, target_col=None):
    print("\n" + "=" * 100)
    print(f"DATASET AUDIT REPORT: {name}")
    print("=" * 100)

    print("\n[1] Dataset Shape")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\n[2] Columns and Data Types")
    print(df.dtypes)

    print("\n[3] Missing Value Analysis")

    missing_count = df.isna().sum()
    missing_percent = (missing_count / len(df)) * 100

    missing_df = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent.round(2)
    }).sort_values("missing_percent", ascending=False)

    print(missing_df[missing_df["missing_count"] > 0])

    if missing_df["missing_count"].sum() == 0:
        print(" No missing values found")


    print("\n[4] Duplicate Row Check")

    dup_count = df.duplicated().sum()
    print(f"Duplicate rows: {dup_count}")



In [36]:
# Call the dataset_audit function for batsman_match
dataset_audit(batsman_match, "Batsman Match Data")


DATASET AUDIT REPORT: Batsman Match Data

[1] Dataset Shape
Rows: 11680
Columns: 23

[2] Columns and Data Types
matchid                           int64
date                     datetime64[ns]
season                           object
venue                            object
city                             object
batting_team                     object
bowling_team                     object
batsman                          object
runs                              int64
balls_faced                       int64
fours                             int64
sixes                             int64
strike_rate                     float64
avg_runs_last_5                 float64
avg_runs_last_10                float64
avg_runs_at_venue               float64
matches_at_venue                  int64
matches_played                    int64
career_avg_runs                 float64
career_avg_balls                float64
boundary_rate                   float64
recent_form_indicator             int64
experie

In [37]:
# Call the dataset_audit function for bowler_match
dataset_audit(bowler_match, "Bowler Match Data")


DATASET AUDIT REPORT: Bowler Match Data

[1] Dataset Shape
Rows: 9283
Columns: 25

[2] Columns and Data Types
matchid                              int64
date                        datetime64[ns]
season                              object
venue                               object
city                                object
bowling_team                        object
batting_team                        object
bowler                              object
runs_conceded                        int64
balls_bowled                         int64
wicket                               int64
wides                                int64
no_balls                             int64
overs                              float64
economy                            float64
avg_wkts_last_5                    float64
avg_wkts_last_10                   float64
avg_wkts_at_venue                  float64
matches_at_venue                     int64
matches_played                       int64
career_avg_wickets           

# Handling the missing values


In [38]:
print("Total missing values in batsman dataset:", batsman_match.isna().sum().sum())

print("\n[CLEANING] Handling missing values in batsman dataset...")

batsman_match.fillna({
    "avg_runs_last_5": 0,
    "avg_runs_last_10": 0,
    "avg_runs_at_venue": 0
}, inplace=True)

print("✅ Missing values handled for batsman dataset")

print("\nRemaining missing values (batsman):")
print(batsman_match.isna().sum()[batsman_match.isna().sum() > 0])

print("Total missing values in batsman dataset:", batsman_match.isna().sum().sum())

Total missing values in batsman dataset: 5573

[CLEANING] Handling missing values in batsman dataset...
✅ Missing values handled for batsman dataset

Remaining missing values (batsman):
Series([], dtype: int64)
Total missing values in batsman dataset: 0


In [39]:
print("Total missing values in bowler dataset:", bowler_match.isna().sum().sum())

print("\n[CLEANING] Handling missing values in bowler dataset...")

bowler_match.fillna({
    "avg_wkts_last_5": 0,
    "avg_wkts_last_10": 0,
    "avg_wkts_at_venue": 0
}, inplace=True)

print("✅ Missing values handled for bowler dataset")

print("\nRemaining missing values (bowler):")
print(bowler_match.isna().sum()[bowler_match.isna().sum() > 0])

print("Total missing values in bowler dataset:", bowler_match.isna().sum().sum())

Total missing values in bowler dataset: 4512

[CLEANING] Handling missing values in bowler dataset...
✅ Missing values handled for bowler dataset

Remaining missing values (bowler):
Series([], dtype: int64)
Total missing values in bowler dataset: 0


In [40]:
import os

print("\n" + "-" * 100)
print("SAVING FINAL DATASETS")
print("-" * 100)

# Create the 'outputs' directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

batsman_match.to_csv("outputs/batsman_match_final_stage2.csv", index=False)
bowler_match.to_csv("outputs/bowler_match_final_stage2.csv", index=False)

print("✅ Files saved successfully")
print("➡ outputs/batsman_match_final_stage2.csv")
print("➡ outputs/bowler_match_final_stage2.csv")


----------------------------------------------------------------------------------------------------
SAVING FINAL DATASETS
----------------------------------------------------------------------------------------------------
✅ Files saved successfully
➡ outputs/batsman_match_final_stage2.csv
➡ outputs/bowler_match_final_stage2.csv
